In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 11 — FUNCTION 4 (CLUSTERING LENS v6)
# Goals vs Week 10:
#  - Explicitly detect promising regions via clustering on top-performing points
#  - Use centroid trend (best cluster centroid) + boundary tightening (cluster radius)
#  - Candidate generation becomes a mixture:
#       (a) local around current best (exploitation)
#       (b) local around best cluster centroid (cluster exploitation)
#       (c) small global spill (guard against missing another basin)
#  - Score adds a "cluster proximity bonus" + "outside-cluster penalty"
#  - Still deterministic: fixed seeds; best-by-score non-duplicate
#  - Enforce domain bounds: x in [0,1]^4
#  - x_next printed to 6 decimals or less
# ============================================================

# ----------------------------
# 0) Helpers
# ----------------------------
def clamp01(a):
    return np.minimum(1.0, np.maximum(0.0, a))

def fmt_x6(x):
    return "[" + ", ".join(f"{float(v):.6f}" for v in x) + "]"

def is_duplicate(x, X_existing, tol=1e-6):
    return np.any(np.linalg.norm(X_existing - x, axis=1) < tol)

def nearest_dist_scaled(cands_scaled, X_scaled_existing):
    diff = cands_scaled[:, None, :] - X_scaled_existing[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1) + 1e-12)

def topk_nearest(x, X, y, k=3):
    d = np.linalg.norm(X - x[None, :], axis=1)
    idx = np.argsort(d)[:k]
    return idx, d[idx], y[idx]

def norm01(v):
    v = np.asarray(v, dtype=np.float64)
    lo, hi = np.min(v), np.max(v)
    return (v - lo) / (hi - lo + 1e-12)

# ----------------------------
# 0b) Simple K-means (no sklearn dependency)
# ----------------------------
def kmeans_simple(X, k=3, seed=42, n_init=6, iters=60):
    rr = np.random.default_rng(seed)
    best_inertia = float("inf")
    best_centers, best_labels = None, None

    n = len(X)
    for _ in range(n_init):
        # kmeans++ init
        centers = np.empty((k, X.shape[1]), dtype=float)
        centers[0] = X[rr.integers(0, n)]
        d2 = np.sum((X - centers[0])**2, axis=1)
        for j in range(1, k):
            p = d2 / (np.sum(d2) + 1e-12)
            centers[j] = X[rr.choice(n, p=p)]
            d2 = np.minimum(d2, np.sum((X - centers[j])**2, axis=1))

        for _it in range(iters):
            dist2 = np.sum((X[:, None, :] - centers[None, :, :])**2, axis=2)
            labels = np.argmin(dist2, axis=1)
            new_centers = np.array([
                X[labels == j].mean(axis=0) if np.any(labels == j) else centers[j]
                for j in range(k)
            ])
            if np.max(np.abs(new_centers - centers)) < 1e-6:
                centers = new_centers
                break
            centers = new_centers

        inertia = float(np.sum((X - centers[labels])**2))
        if inertia < best_inertia:
            best_inertia = inertia
            best_centers, best_labels = centers.copy(), labels.copy()

    return best_centers, best_labels, best_inertia

# ----------------------------
# 1) Input data (Function 4) — RAW
# ----------------------------
X_train_raw = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.8893564 , 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.0062504 , 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.7570915 , 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.6260706 , 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.2119651 , 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.9027701 , 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.4723669 , 0.453192  , 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.8548108 , 0.49396462, 0.73530997, 0.80809201],
    [1.085621  , 1.019592  , 1.039177  , 1.099482  ],   # out-of-bounds in raw
    [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
    [0.866175  , 0.601115  , 0.708072  , 0.020585  ],
    [0.145904  , 0.536548  , 0.6014    , 0.01905   ],
    [0.356293  , 0.442523  , 0.13052   , 0.242559  ],
    [0.061431  , 0.381247  , 0.983792  , 0.705575  ],
    [0.497045, 0.450388, 0.380113, 0.297612],
    [0.480706, 0.444032, 0.354963, 0.354729],
    [0.503198, 0.435617, 0.371338, 0.408316],
    [0.506271, 0.414408, 0.366165, 0.398853]
], dtype=float)

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.60493430274798, -22.782193418373407, -22.194212794446454, -13.363105653346768,  -5.926020577803715,
    -23.786955955997737, -2.5615259470796796, -0.81371612670717, -1.1642621740684471, -1.2544758182228928
], dtype=float)

assert len(X_train_raw) == len(y_train), "X_train and y_train length mismatch"

# ----------------------------
# 2) Enforce domain bounds [0,1]^4
# ----------------------------
X_train = clamp01(X_train_raw)

# ----------------------------
# 3) Current best (maximisation)
# ----------------------------
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X (clamped to [0,1]):", current_best_x)
print("Current best y:", current_best_y)

# ----------------------------
# 4) Fixed scaling for [0,1]^4 (raw == scaled)
# ----------------------------
X_scaled = X_train.copy()
y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

# ----------------------------
# 5) Surrogate model (kept) — but slightly lighter for Week 11 stability
# ----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=4, hidden=(64, 64), p_dropout=0.08):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_model(model, X, y, max_epochs=1000, lr=1e-3, weight_decay=1e-5, patience=70, min_delta=1e-4):
    criterion = nn.SmoothL1Loss(beta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best = float("inf")
    bad = 0
    model.train()
    for _ in range(max_epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        lv = float(loss.item())
        if lv < best - min_delta:
            best = lv
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    return best

# ----------------------------
# 6) Train a small bootstrapped ensemble (Week 11: fewer models, more stable)
# ----------------------------
def train_ensemble(X_tensor, y_tensor, rr, n_ens=7, hidden=(64,64), dropout=0.08):
    ensemble = []
    n = len(X_tensor)
    for m in range(n_ens):
        boot_idx = rr.integers(0, n, size=n)
        Xb = X_tensor[boot_idx]
        yb = y_tensor[boot_idx]
        torch.manual_seed(500 + m)
        model = MLP(input_dim=4, hidden=hidden, p_dropout=dropout).to(device)
        train_model(model, Xb, yb, max_epochs=1000, lr=1e-3, weight_decay=1e-5, patience=70)
        ensemble.append(model)
    return ensemble

ensemble = train_ensemble(X_tensor_all, y_tensor_all, rng, n_ens=7)

# ----------------------------
# 7) Prediction utilities (original y units) + EI/PI
# ----------------------------
def ensemble_predict(ensemble, X_scaled_tensor):
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X_scaled_tensor).squeeze(-1)
            p_raw = p_scaled * y_std + y_mean
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)
    return preds.mean(dim=0), preds.std(dim=0) + 1e-9

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.006):
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    return torch.where(std > 0, ei, torch.zeros_like(ei))

def probability_of_improvement(mean, std, best_y, xi=0.0):
    imp = mean - best_y - xi
    Z = imp / std
    return normal.cdf(Z)

# ----------------------------
# 8) WEEK 11: cluster detection on promising region
#    We cluster TOP-Q points (by y), then pick the best cluster by max(y).
# ----------------------------
TOP_Q_FRAC = 0.25  # top 25% are "signal"; rest treated more like noise
k_clusters = 3

n_top = max(6, int(np.ceil(TOP_Q_FRAC * len(X_train))))
top_idx = np.argsort(y_train)[::-1][:n_top]
X_top = X_train[top_idx]
y_top = y_train[top_idx]

centers, labels, inertia = kmeans_simple(X_top, k=k_clusters, seed=42)

# pick "best cluster" by its max observed y
best_c = None
best_c_maxy = -float("inf")
for c in range(k_clusters):
    members = (labels == c)
    if not np.any(members):
        continue
    c_maxy = float(np.max(y_top[members]))
    if c_maxy > best_c_maxy:
        best_c_maxy = c_maxy
        best_c = c

cluster_centroid = centers[best_c]
cluster_members = X_top[labels == best_c]

# boundary tightening: cluster radius based on 90th percentile distance to centroid
d_to_centroid = np.linalg.norm(cluster_members - cluster_centroid[None, :], axis=1)
cluster_r90 = float(np.quantile(d_to_centroid, 0.90) + 1e-12)

print("\n=== WEEK 11 CLUSTER SUMMARY ===")
print("Top-Q points used:", n_top)
print("Chosen best cluster id:", int(best_c))
print("Cluster centroid:", fmt_x6(cluster_centroid))
print("Cluster r90 boundary:", f"{cluster_r90:.6f}")
print("Best y inside cluster (max):", f"{best_c_maxy:.6f}")

# ----------------------------
# 9) WEEK 11 candidate search:
#    mixture = (local-best) + (local-centroid) + (global)
#    score = mean/EI/PI - dist_penalty + cluster_bonus - outside_cluster_penalty
# ----------------------------
DECODING = {"max_tokens": 9000}  # smaller = more focused & reproducible

CANDIDATE_MIX = {
    "best_local_frac": 0.70,
    "cluster_local_frac": 0.20,
    "global_frac": 0.10,
    "trust_radius_best": 0.07,      # tightened vs Week 10
    "trust_radius_cluster": 0.045,  # even tighter around centroid
}

REFINE = {"topk_seed": 220, "per_seed": 18, "refine_radius": 0.040}

SCORE_WEIGHTS = {"w_mean": 0.72, "w_ei": 0.23, "w_pi": 0.05}
PENALTY = {
    "lambda_dist": 0.10,          # extrapolation risk penalty (nearest observed)
    "w_cluster_bonus": 0.08,      # prefer dense promising region
    "lambda_outside": 0.10,       # boundary tightening (penalize leaving cluster)
}

def propose_next_point_week11_clustered(
    ensemble, X_existing_raw01, X_scaled_existing,
    best_x_raw01, cluster_centroid_raw01, cluster_r90,
    rr, xi=0.006, dup_tol=1e-6,
    decoding=DECODING, cand_mix=CANDIDATE_MIX, refine=REFINE,
    weights=SCORE_WEIGHTS, penalty=PENALTY, top_report=10
):
    n_total = int(decoding["max_tokens"])

    n_best = int(cand_mix["best_local_frac"] * n_total)
    n_clu  = int(cand_mix["cluster_local_frac"] * n_total)
    n_glo  = n_total - n_best - n_clu

    # (a) local around current best
    rb = float(cand_mix["trust_radius_best"])
    local_best = best_x_raw01 + rr.normal(0.0, rb, size=(n_best, 4)).astype(np.float32)
    local_best = clamp01(local_best)

    # (b) local around best cluster centroid
    rc = float(cand_mix["trust_radius_cluster"])
    local_cluster = cluster_centroid_raw01 + rr.normal(0.0, rc, size=(n_clu, 4)).astype(np.float32)
    local_cluster = clamp01(local_cluster)

    # (c) small global coverage
    global_c = rr.random((n_glo, 4), dtype=np.float32)

    coarse_scaled = np.vstack([local_best, local_cluster, global_c]).astype(np.float32)

    # ----- coarse scoring -----
    X_cand_tensor = torch.tensor(coarse_scaled, dtype=torch.float32, device=device)
    mean, std = ensemble_predict(ensemble, X_cand_tensor)

    best_y = float(np.max(y_train))
    ei = expected_improvement(mean, std, best_y, xi=xi)
    pi = probability_of_improvement(mean, std, best_y, xi=0.0)

    mean_np = mean.detach().cpu().numpy()
    std_np  = std.detach().cpu().numpy()
    ei_np   = ei.detach().cpu().numpy()
    pi_np   = pi.detach().cpu().numpy()

    mean_n = norm01(mean_np)
    ei_n   = norm01(ei_np)
    pi_n   = norm01(pi_np)

    # nearest-observed distance penalty (extrapolation risk)
    d_near = nearest_dist_scaled(coarse_scaled.astype(np.float64), X_scaled_existing.astype(np.float64))
    d_n = norm01(d_near)

    # cluster proximity bonus: prefer points close to centroid
    d_cent = np.linalg.norm(coarse_scaled - cluster_centroid_raw01[None, :], axis=1)
    d_cent_n = norm01(d_cent)
    cluster_bonus = (1.0 - d_cent_n)  # closer => larger bonus

    # outside-cluster boundary tightening penalty
    # penalize if beyond r90 (in raw/0-1 space); soft hinge
    outside = np.maximum(0.0, (d_cent - cluster_r90))
    outside_n = norm01(outside)

    score = (
        weights["w_mean"] * mean_n +
        weights["w_ei"]   * ei_n +
        weights["w_pi"]   * pi_n -
        float(penalty["lambda_dist"]) * d_n +
        float(penalty["w_cluster_bonus"]) * cluster_bonus -
        float(penalty["lambda_outside"]) * outside_n
    )

    # top-k seeds
    topk = int(refine["topk_seed"])
    seed_idx = np.argsort(score)[::-1][:topk]
    seeds = coarse_scaled[seed_idx]

    # ----- refinement -----
    per_seed = int(refine["per_seed"])
    rr_ref = float(refine["refine_radius"])

    refine_points = []
    for s in seeds:
        jitter = s + rr.normal(0.0, rr_ref, size=(per_seed, 4)).astype(np.float32)
        jitter = clamp01(jitter)
        refine_points.append(jitter)

    refine_scaled = np.vstack([coarse_scaled] + refine_points).astype(np.float32)

    X_ref_tensor = torch.tensor(refine_scaled, dtype=torch.float32, device=device)
    mean2, std2 = ensemble_predict(ensemble, X_ref_tensor)

    ei2 = expected_improvement(mean2, std2, best_y, xi=xi)
    pi2 = probability_of_improvement(mean2, std2, best_y, xi=0.0)

    mean2_np = mean2.detach().cpu().numpy()
    std2_np  = std2.detach().cpu().numpy()
    ei2_np   = ei2.detach().cpu().numpy()
    pi2_np   = pi2.detach().cpu().numpy()

    mean2_n = norm01(mean2_np)
    ei2_n   = norm01(ei2_np)
    pi2_n   = norm01(pi2_np)

    d2_near = nearest_dist_scaled(refine_scaled.astype(np.float64), X_scaled_existing.astype(np.float64))
    d2_n = norm01(d2_near)

    d2_cent = np.linalg.norm(refine_scaled - cluster_centroid_raw01[None, :], axis=1)
    d2_cent_n = norm01(d2_cent)
    cluster_bonus2 = (1.0 - d2_cent_n)

    outside2 = np.maximum(0.0, (d2_cent - cluster_r90))
    outside2_n = norm01(outside2)

    score2 = (
        weights["w_mean"] * mean2_n +
        weights["w_ei"]   * ei2_n +
        weights["w_pi"]   * pi2_n -
        float(penalty["lambda_dist"]) * d2_n +
        float(penalty["w_cluster_bonus"]) * cluster_bonus2 -
        float(penalty["lambda_outside"]) * outside2_n
    )

    # Filter to non-duplicates; greedy best-by-score
    keep = []
    for i in range(len(refine_scaled)):
        x_raw01 = refine_scaled[i]
        if not is_duplicate(x_raw01, X_existing_raw01, tol=dup_tol):
            keep.append(i)

    if len(keep) == 0:
        chosen_i = int(np.argmax(score2))
    else:
        keep = np.array(keep, dtype=int)
        chosen_i = int(keep[np.argmax(score2[keep])])

    def pack(j):
        xr01 = refine_scaled[j].astype(np.float64)
        return {
            "idx": int(j),
            "x_raw01": xr01,
            "mean": float(mean2_np[j]),
            "std": float(std2_np[j]),
            "ei": float(ei2_np[j]),
            "pi": float(pi2_np[j]),
            "score": float(score2[j]),
            "d_near": float(d2_near[j]),
            "d_cent": float(d2_cent[j]),
            "outside": float(outside2[j]),
        }

    chosen = pack(chosen_i)

    pool = np.arange(len(refine_scaled)) if len(keep) == 0 else keep
    pool_sorted_score = pool[np.argsort(score2[pool])[::-1]]
    pool_sorted_ei = pool[np.argsort(ei2_np[pool])[::-1]]

    report = {
        "top_by_ei": [pack(j) for j in pool_sorted_ei[:top_report]],
        "top_by_score": [pack(j) for j in pool_sorted_score[:top_report]],
    }

    meta = {
        "best_y": best_y,
        "n_candidates_total": int(n_total),
        "n_candidates_refined": int(len(refine_scaled)),
        "cand_mix": cand_mix,
        "refine": refine,
        "weights": weights,
        "penalty": penalty,
        "selection": "GREEDY_BEST_BY_SCORE + CLUSTER_BONUS + BOUNDARY_TIGHTENING",
        "domain_bounds": "[0,1]^4 (enforced)",
        "cluster_centroid": cluster_centroid_raw01,
        "cluster_r90": cluster_r90
    }
    return chosen, report, meta

chosen, report, meta = propose_next_point_week11_clustered(
    ensemble=ensemble,
    X_existing_raw01=X_train,
    X_scaled_existing=X_scaled,
    best_x_raw01=current_best_x.astype(np.float32),
    cluster_centroid_raw01=cluster_centroid.astype(np.float32),
    cluster_r90=cluster_r90,
    rr=rng,
    xi=0.006,
    dup_tol=1e-6,
    top_report=10
)

next_x = clamp01(chosen["x_raw01"])
next_mean = chosen["mean"]
next_std  = chosen["std"]
next_ei   = chosen["ei"]
next_pi   = chosen["pi"]
next_score = chosen["score"]

# ----------------------------
# 10) Interpretability add-ons
# ----------------------------
nn_idx, nn_dist, nn_y = topk_nearest(next_x, X_train, y_train, k=3)

# ----------------------------
# 11) Report
# ----------------------------
print("\n================ WEEK 11 FUNCTION 4 RESULTS (v6 — CLUSTERING LENS) ================")

print("\nCURRENT BEST OBSERVED")
print("x_best =", fmt_x6(current_best_x), ", y_best =", f"{current_best_y:.6f}")

print("\nWEEK 11 SETTINGS (Clustering lens)")
print("Domain bounds:", meta["domain_bounds"])
print("Candidate mix:", meta["cand_mix"])
print("Refinement:", meta["refine"])
print("Score weights:", meta["weights"])
print("Penalties:", meta["penalty"])
print("Selection:", meta["selection"])
print("Cluster centroid:", fmt_x6(meta["cluster_centroid"]))
print("Cluster r90:", f"{meta['cluster_r90']:.6f}")
print("Total coarse candidates:", meta["n_candidates_total"])
print("Total evaluated after refinement:", meta["n_candidates_refined"])

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by SCORE used for selection)")
for i, r in enumerate(report["top_by_score"], 1):
    print(
        f"{i:02d}) x={fmt_x6(r['x_raw01'])} | mean={r['mean']:.6f} std={r['std']:.6f} "
        f"EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f} "
        f"d_near={r['d_near']:.6f} d_cent={r['d_cent']:.6f} outside={r['outside']:.6f}"
    )

print("\nRECOMMENDED NEXT POINT (domain-safe; x_next in 6 decimals)")
print("x_next     =", fmt_x6(next_x))
print("mu(x_next) =", f"{next_mean:.6f}")
print("sigma      =", f"{next_std:.6f}")
print("EI         =", f"{next_ei:.6f}")
print("PI         =", f"{next_pi:.6f}")
print("SCORE      =", f"{next_score:.6f}")

print("\nINTERPRETABILITY CHECKS")
print("Nearest observed points to x_next (context):")
for rank, (ii, dd, yy) in enumerate(zip(nn_idx, nn_dist, nn_y), 1):
    print(f"  {rank}) idx={int(ii)}  x={fmt_x6(X_train[int(ii)])}  y={float(yy):.6f}  dist={float(dd):.6f}")

# ----------------------------
# 12) Week 11 reasoning prompts (explicit)
# ----------------------------
print("\nWEEK 11 REASONING (Clustering Techniques)")
print("- Pattern observed: the best outputs occur in a tight local group (dense region) formed by recent samples.")
print("- Cluster step: we clustered the top-performing points and selected the best cluster by max(y).")
print("- Centroid trend: we exploit around the cluster centroid (a representative 'center' of the promising basin).")
print("- Boundary tightening: we penalize candidates that drift beyond the cluster’s r90 radius (noise / extrapolation).")
print("- Adjustment vs less-effective choices: reduced pure-local-only bias by adding explicit centroid-driven sampling.")
print("- If plotted: you’d see a dense high-performing cluster plus scattered lower-performing regions; Week 11 focuses on the dense cluster while keeping a small global spill.")



Current best index: 37
Current best X (clamped to [0,1]): [0.480706 0.444032 0.354963 0.354729]
Current best y: -0.81371612670717

=== WEEK 11 CLUSTER SUMMARY ===
Top-Q points used: 10
Chosen best cluster id: 1
Cluster centroid: [0.486880, 0.435957, 0.338154, 0.325179]
Cluster r90 boundary: 0.203279
Best y inside cluster (max): -0.813716

================ WEEK 11 FUNCTION 4 RESULTS (v6 — CLUSTERING LENS) ================

CURRENT BEST OBSERVED
x_best = [0.480706, 0.444032, 0.354963, 0.354729] , y_best = -0.813716

WEEK 11 SETTINGS (Clustering lens)
Domain bounds: [0,1]^4 (enforced)
Candidate mix: {'best_local_frac': 0.7, 'cluster_local_frac': 0.2, 'global_frac': 0.1, 'trust_radius_best': 0.07, 'trust_radius_cluster': 0.045}
Refinement: {'topk_seed': 220, 'per_seed': 18, 'refine_radius': 0.04}
Score weights: {'w_mean': 0.72, 'w_ei': 0.23, 'w_pi': 0.05}
Penalties: {'lambda_dist': 0.1, 'w_cluster_bonus': 0.08, 'lambda_outside': 0.1}
Selection: GREEDY_BEST_BY_SCORE + CLUSTER_BONUS + BOUNDA